<a href="https://colab.research.google.com/github/Santhosh200429/Multi-Agent-Customer-Support-Assistant-/blob/main/Multi_Agent_Customer_Support_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This installs Java 17 on Colab

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk -qq


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected

Install Gson (for JSON output)

In [ ]:
!wget https://repo1.maven.org/maven2/com/google/code/gson/gson/2.10.1/gson-2.10.1.jar -q


This implements a simple Multi-Agent Customer Support Assistant designed for enterprise workflows. It uses three lightweight agents—an Intent Agent, a Reply Agent, and an Escalation Agent—coordinated by a central Orchestrator. Together, they classify the user’s message, generate a professional reply, and decide whether escalation is required. The system demonstrates clean multi-agent orchestration suitable for enterprise automation tasks such as ticket handling, customer assistance, and workflow optimization.

In [ ]:
code = r"""
import java.time.LocalDateTime;
import java.util.*;
import com.google.gson.Gson;
import com.google.gson.GsonBuilder;

// ---------------------- Memory Class ----------------------
class Memory {

    static class Message {
        String role;
        String content;
        String time;

        Message(String role, String content) {
            this.role = role;
            this.content = content;
            this.time = LocalDateTime.now().toString();
        }
    }

    private List<Message> messages = new ArrayList<>();
    private final int maxHistory = 20;

    public void add(String role, String content) {
        messages.add(new Message(role, content));

        if (messages.size() > maxHistory) {
            messages = messages.subList(messages.size() - maxHistory, messages.size());
        }
    }

    public String getContext() {
        StringBuilder out = new StringBuilder();
        int start = Math.max(messages.size() - 5, 0);

        for (int i = start; i < messages.size(); i++) {
            Message m = messages.get(i);
            out.append(m.role).append(": ").append(m.content).append("\n");
        }
        return out.toString();
    }
}

// ---------------------- Intent Classifier Agent ----------------------
class IntentAgent {

    public String[] classify(String message) {
        String text = message.toLowerCase();

        if (text.contains("refund")) {
            return new String[]{"refund", "high"};
        }
        if (text.contains("cancel")) {
            return new String[]{"cancellation", "high"};
        }
        if (text.contains("invoice") || text.contains("bill")) {
            return new String[]{"billing", "medium"};
        }
        if (text.contains("help")) {
            return new String[]{"general_help", "low"};
        }
        return new String[]{"general", "low"};
    }
}

// ---------------------- Reply Generator Agent ----------------------
class ReplyAgent {

    public String createReply(String message, String intent, String urgency) {

        switch (intent) {
            case "refund":
                return "I understand you want a refund. Please share your order ID so I can assist you further.";

            case "cancellation":
                return "I can help you cancel your subscription. Kindly provide your registered email.";

            case "billing":
                return "It seems you have a billing concern. Please send your invoice number for verification.";

            case "general_help":
                return "Sure, I'm here to help. Could you please share more details?";

            default:
                return "Thank you for your message. How can I assist you today?";
        }
    }
}

// ---------------------- Escalation Agent ----------------------
class EscalationAgent {

    public Map<String, Object> check(String intent, String urgency, String message) {
        Map<String, Object> result = new HashMap<>();

        if ("high".equals(urgency)) {
            result.put("escalate", true);
            result.put("note", "Urgent " + intent + " issue. Needs human review.");
        } else {
            result.put("escalate", false);
            result.put("note", "No escalation required.");
        }

        return result;
    }
}

// ---------------------- Coordinator Agent ----------------------
class Coordinator {

    private IntentAgent intentAgent = new IntentAgent();
    private ReplyAgent replyAgent = new ReplyAgent();
    private EscalationAgent escalationAgent = new EscalationAgent();
    private Memory memory = new Memory();

    public Map<String, Object> ask(String message) {

        memory.add("user", message);

        String[] result = intentAgent.classify(message);
        String intent = result[0];
        String urgency = result[1];

        String reply = replyAgent.createReply(message, intent, urgency);
        Map<String, Object> escalation = escalationAgent.check(intent, urgency, message);

        Map<String, Object> finalOutput = new LinkedHashMap<>();
        finalOutput.put("intent", intent);
        finalOutput.put("urgency", urgency);
        finalOutput.put("reply", reply);
        finalOutput.put("escalation", escalation);

        memory.add("agent", reply);
        return finalOutput;
    }
}

// ---------------------- Main Program (Testing) ----------------------
public class MultiAgentSystem {

    public static void main(String[] args) {

        Coordinator agent = new Coordinator();
        Gson gson = new GsonBuilder().setPrettyPrinting().create();

        List<String> messages = Arrays.asList(
                "I want to cancel my subscription.",
                "My invoice amount is wrong.",
                "I need a refund please.",
                "Hello, I need help."
        );

        for (String msg : messages) {
            System.out.println("USER: " + msg);
            Map<String, Object> out = agent.ask(msg);
            System.out.println(gson.toJson(out));
            System.out.println("--------------------------------------------------");
        }
    }
}
"""

with open("MultiAgentSystem.java", "w") as f:
    f.write(code)


Compile and Run

In [ ]:
!javac -cp gson-2.10.1.jar MultiAgentSystem.java
!java -cp ".:gson-2.10.1.jar" MultiAgentSystem


USER: I want to cancel my subscription.
{
  "intent": "cancellation",
  "urgency": "high",
  "reply": "I can help you cancel your subscription. Kindly provide your registered email.",
  "escalation": {
    "note": "Urgent cancellation issue. Needs human review.",
    "escalate": true
  }
}
--------------------------------------------------
USER: My invoice amount is wrong.
{
  "intent": "billing",
  "urgency": "medium",
  "reply": "It seems you have a billing concern. Please send your invoice number for verification.",
  "escalation": {
    "note": "No escalation required.",
    "escalate": false
  }
}
--------------------------------------------------
USER: I need a refund please.
{
  "intent": "refund",
  "urgency": "high",
  "reply": "I understand you want a refund. Please share your order ID so I can assist you further.",
  "escalation": {
    "note": "Urgent refund issue. Needs human review.",
    "escalate": true
  }
}
--------------------------------------------------
USER: Hell